# Categorical-profile analysis (no-alarm)

Contract: [`profile_numeric_shift_plan.md`](../sample_space_partitioning/feature_analysis/profile_numeric_shift_plan.md).

Scope is **no-alarm train** only. For each observed cell of P1–P9 the package records (1) a label census, (2) a label-blind numeric shift versus the nested parent, (3) within-cell compactness of frauds against non-frauds on the shared quantile map $q = F_{\mathrm{cell}}(X)$.

Machinery lives in `ccf_ml.data_analysis`. This notebook loads the artifacts, prints the tables that decide the story, and does not re-implement the loop. No training, no OOF, no `merchant_tier`, no `day_of_week`.

In [ ]:
%load_ext autoreload
%autoreload 2

from ccf_ml.data_analysis import (
    NOTEBOOK_STEM,
    artifacts_exist,
    load_tables,
    run_cat_profiling,
)
from ccf_ml.data_analysis.constants import DISTANCE_COLUMN
from ccf_ml.data_analysis.profiles import load_no_alarm_train
from ccf_ml.cross_validation.constants import TARGET_COLUMN
from IPython.display import display
import pandas as pd

region = load_no_alarm_train()
n = len(region)
n_fraud = int(region[TARGET_COLUMN].sum())
print(f"no_alarm rows={n} frauds={n_fraud} rate={n_fraud / n:.4%}")
print(f"artifacts exist: {artifacts_exist(script_name=NOTEBOOK_STEM)}")

tables = run_cat_profiling(frame=region, script_name=NOTEBOOK_STEM, force=False)
census = tables.census
parent_shift = tables.parent_shift
compact_cell = tables.compactness_cell
print("census", census.shape, "shift", parent_shift.shape, "compact_cell", compact_cell.shape)

## Measurement 1 — label census

Tags are pre-registered: **bulky_cold** if $n_{\mathrm{fraud}} \ge 100$ and rate $\le$ region rate; **hot** if lift $\ge 2$ and $n_{\mathrm{fraud}} \ge 20$. They are not a filter on measurements 2–3.

In [ ]:
tag_counts = census.groupby(["profile_id", "tag"]).size().unstack(fill_value=0)
display(tag_counts)

print("bulky_cold")
bulky = census.loc[census["tag"] == "bulky_cold"].sort_values(
    ["n_fraud", "n"], ascending=False
)
display(bulky.head(40))

print("hot")
hot = census.loc[census["tag"] == "hot"].sort_values("lift", ascending=False)
display(hot.head(40))

p6 = census.loc[census["profile_id"] == "P6"].sort_values("n_fraud", ascending=False)
print(f"P6 observed cells: {len(p6)} (theoretical 40; missing pairs are absent)")
display(p6)

## Measurement 3 — compactness inside the cell

Shared cell CDF, then $d_{ij} = q^1_i - q^0_j$. Compactness is the median over frauds of mean 5-NN $|d|$. A useful numeric **breaks** compactness for a non-trivial share of frauds, not merely one with $|\mathrm{mean}(d)| \ge 0.10$.

P6 × `distance_from_home_km` is the running example (computer × a channel, far from home).

In [ ]:
p6_dist = compact_cell.loc[
    (compact_cell["profile_id"] == "P6")
    & (compact_cell["numeric"] == DISTANCE_COLUMN)
].sort_values("median_nn5")
print("P6 distance compactness")
display(
    p6_dist.loc[
        :,
        [
            "cell_key",
            "n_fraud",
            "n_nonfraud",
            "mean_d",
            "auc",
            "median_nn5",
            "share_almost_double",
            "share_separated",
            "min_d_global",
            "max_d_global",
            "tag",
        ],
    ]
)

p6_joint = compact_cell.loc[
    (compact_cell["profile_id"] == "P6")
    & (compact_cell["numeric"].astype(str).str.startswith("joint:"))
].sort_values("median_nn5")
print("P6 joint compactness")
display(
    p6_joint.loc[
        :,
        [
            "cell_key",
            "numeric",
            "n_fraud",
            "median_nn5",
            "share_almost_double",
            "tag",
        ],
    ]
)

print("least compact cells (any profile, 1-d)")
one_d = compact_cell.loc[~compact_cell["numeric"].astype(str).str.startswith("joint:")]
display(
    one_d.sort_values("median_nn5", ascending=False).head(25).loc[
        :,
        [
            "profile_id",
            "spec_name",
            "cell_key",
            "numeric",
            "median_nn5",
            "mean_d",
            "share_separated",
            "share_almost_double",
            "tag",
        ],
    ]
)

## Measurement 2 — numeric vs parent

Hit if both sides have at least 500 rows and $D = \sup |F_{\mathrm{cell}} - F_{\mathrm{parent}}| \ge 0.05$. P7–P9 must also beat $1.1\times$ a scramble of the window axis inside the parent; at most one window per `(parent slice, numeric)` is distilled. P1 hits are `descriptive_only`.

In [ ]:
print(parent_shift["verdict"].value_counts())
print("distilled", int(parent_shift["distilled"].astype(bool).sum()))

fe = parent_shift.loc[parent_shift["verdict"] == "fe_candidate"].sort_values(
    "ks_d", ascending=False
)
print("fe_candidate")
display(
    fe.head(40).loc[
        :,
        [
            "profile_id",
            "spec_name",
            "cell_key",
            "parent_axis",
            "numeric",
            "ks_d",
            "delta_f_p50",
            "delta_f_p90",
            "mean_parent_rank",
            "scramble_d",
            "distilled",
        ],
    ]
)

distilled = parent_shift.loc[parent_shift["distilled"].astype(bool)].sort_values(
    "ks_d", ascending=False
)
print("distilled windows")
display(distilled)

## Takeaway

From the no-alarm run (`646938` rows, `6931` frauds):

- **Compactness holds.** Every 1-d cell has median 5-NN $|d| < 0.01$ (max $0.0034$). About $95\%$ of cells have `share_separated` $= 0$. Velocity can show $|\mathrm{mean}(d)| \ge 0.10$ while every fraud still has a near-double — the pair lens is doing the job aggregates would miss.
- **P6 is fully observed** (`40 / 40` device $\times$ channel cells). There is no `computer` level; the desktop stand-ins are `Windows PC` and `Mac`. On `distance_from_home_km`, P6 mean $d$ is near $0$ (Windows PC$|$Online $0.030$), `share_almost_double` $= 1$, and no fraud sits above every non-fraud. Distance vs parent is `below_floor` in every P6 cell. Far-from-home inside a device $\times$ channel cell does not isolate no-alarm frauds.
- **Vs-parent `pq_` leftovers are thin.** P2/P3 never hit $D \ge 0.05$ against the merchant parent. P7 (hour) and P9 (device $\times$ age) have no `fe_candidate`. The large P4/P5 $D$ on `amount_usd` is merchant scale (Crypto / Travel / Electronics) reappearing inside a device or channel — the tree already has `merchant_category`. Distilled windows: three P8 cells only (Crypto Exchange ages $[44, 52)$ on amount; Gift Cards and Crypto Exchange ages $[18, 26)$ on velocity), all $D$ just above $0.05$.
- **Label census.** Hot 2-ways are the known ones: Gift Cards / Gaming / Crypto $\times$ `No Authentication` or `is_new_merchant=True` (lift $9$--$17$). Bulky_cold is mostly large low-rate slices (`is_new_merchant=False`, Online Retail, Android Phone).

A later OOF (not this pass) can try `pq_` on those three distilled P8 windows. Do not add `profile_id` dummies. Compactness breaking is not a reason to distill a numeric here — it did not break.